In [ ]:
import pandas as pd
import numpy as np
import json

#  Calendar year to Financial year map
fy_map = lambda q: q.year - 1 if q.quarter == 1 else q.year

fc = pd.read_csv("../../data/outputs/forecasts/obr_scenario_forecasts.csv")
p = json.load(open("../../data/processed/bridge_params.json"))
fc["ensemble_log"] = (fc["ardl_log"] + fc["nardl_log"]) / 2

# Seed with last 4 actual log starts from England master
eng = pd.read_csv("../../data/python_master/england_master.csv")
ln_S_full = np.concatenate([np.log(eng["starts"].dropna().iloc[-4:].values), fc["ensemble_log"].values])
quarters = pd.PeriodIndex(fc["period"], freq="Q")
seasonal = {1: 0, 2: p["q2"], 3: p["q3"], 4: p["q4"]}

ln_C, lag_C = [], p["last_ln_C"]

for t in range(len(fc)):
    lag_C = p["intercept"] + p["rho"]*lag_C + p["beta"]*ln_S_full[t] + seasonal[quarters[t].quarter]
    ln_C.append(lag_C)
fc["completions"] = np.exp(ln_C) * p["smearing_factor"]
fc["fy"] = quarters.map(fy_map)
annual = fc.groupby("fy")["completions"].sum().rename("private_completions").to_frame().iloc[1:]
display(annual.head())

In [ ]:
def parse_quarter(x):
    x = str(x)
    qmap = {"Jan - Mar": 1, "Apr - Jun": 2, "Jul - Sep": 3, "Oct - Dec": 4}
    for k, v in qmap.items():
        if x.startswith(k):
            try: return pd.Period(year=int(x[-4:]), quarter=v, freq="Q")
            except: return pd.NaT
    return pd.NaT

ons = pd.read_excel("../../data/raw/starts/indicatorsofukhousebuilding.xlsx", sheet_name="1b", skiprows=5)
ons["quarter"] = ons["Period"].apply(parse_quarter)
ons = ons.dropna(subset=["quarter"]).set_index("quarter").sort_index()
ons_fy = ons.index.map(fy_map)

# LT120 components
components = ["New build completions", "Net conversions", "Net change of use",
             "Net other gains", "Demolitions", "Total net additional dwellings"]
lt120 = pd.read_excel("../../data/raw/net_additions/Live_Table_120.ods", sheet_name="LT120_unrounded", skiprows=4)
lt120 = lt120.set_index(lt120.columns[0]).T[components].iloc[:-2].apply(pd.to_numeric, errors="coerce")
avg = lt120.iloc[-4:-1].mean()

# Non-private new build = Table 120 total - actual private 
priv_actual = ons.groupby(ons_fy)["Completed - Private Enterprise"].sum().loc[2021:2023].mean()
non_private = avg["New build completions"] - priv_actual

# Net additions identity
net_add = lambda priv: (priv + non_private + avg["Net conversions"] + avg["Net change of use"]
                        - avg["Demolitions"] + avg["Net other gains"])
annual["net_additions"] = net_add(annual["private_completions"])

# FY2025-26: actual back-quarters + 1 forecast quarter 
actual_back = ons.loc[pd.PeriodIndex(["2025Q2","2025Q3","2025Q4"], freq="Q"), "Completed - Private Enterprise"].sum()
priv_2025_26 = actual_back + fc.loc[fc["period"] == "2026Q1", "completions"].iloc[0]

# Parliament tally (FY2024-25 actual + four modelled years)
delivery = {"2024-25": 208600, "2025-26": net_add(priv_2025_26),
            "2026-27": annual.loc[2026, "net_additions"],
            "2027-28": annual.loc[2027, "net_additions"],
            "2028-29": annual.loc[2028, "net_additions"]}

cumulative, target = sum(delivery.values()), 1_500_000
for fy, v in delivery.items():
    print(f"{fy}: {v:,.0f}")
print(f"\nCumulative: {cumulative:,.0f} ({100*cumulative/target:.1f}% of {target:,}, shortfall {target-cumulative:,.0f})")

non-private new build is held flat at its recent average (it won't respond to the policy scenarios), and conversions/change-of-use/demolitions are likewise held at 2021-24 averages. 